# Diamond frontier4-cache: WikiText-103 + FineWeb-Edu (Kaggle, 2h)

This notebook trains the frontier4-cache architecture for **2 hours on one Kaggle GPU** using:

- **WikiText-103 raw** (SHA-pinned NaNROUTER release)
- **FineWeb-Edu** (`HuggingFaceFW/fineweb-edu`, `sample-10BT`, streamed)

Kaggle setup:

1. **Settings → Accelerator → GPU T4 x1** (or P100).
2. **Settings → Internet → On**.
3. Run the cells in order.

State is written to `/kaggle/working/diamond_state` after every epoch. A 2-hour session stops gracefully; rerun the training cell to resume within the same session. To resume in a new session, download/upload `diamond_state.npz` and `diamond_history.jsonl`.

The default model is the larger `large` preset (~30M parameters). If it runs out of memory, set `MODEL_SIZE=frontier` or `BATCH=16` in the training cell.


In [ ]:
#@title Check GPU / TensorFlow and install FineWeb streaming deps
!nvidia-smi || true
!pip -q install datasets
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPUs", tf.config.list_physical_devices("GPU"))
# If the engine hits a TF/Keras API mismatch, install the tested stack and restart:
# !pip -q install "tensorflow==2.16.2" "keras==3.0.5" "numpy==1.26.4"


In [ ]:
#@title Clone the Kaggle branch
%cd /kaggle/working
!rm -rf nanlm_experimental
!git clone --branch kaggle-fineweb --depth 1 https://github.com/shemshallah/nanlm_experimental.git
%cd /kaggle/working/nanlm_experimental
!git log --oneline -1


In [ ]:
#@title Download and split WikiText-103
!if [ ! -s /kaggle/working/wiki.train.raw ]; then \
  wget -q --show-progress -O /kaggle/working/wiki.train.raw \
    https://github.com/shemshallah/NaNROUTER/releases/download/wikitext103/wiki.train.raw; \
fi
!python colab/prepare_wikitext.py \
  --raw /kaggle/working/wiki.train.raw \
  --out /kaggle/working/corpus \
  --max-articles 4000 \
  --max-chars 80000000


In [ ]:
#@title Stream a FineWeb-Edu subset into the corpus (15-minute cap)
# If HuggingFace streaming stalls, continue with WikiText-only rather than
# hanging the whole kernel.
!timeout 900 python kaggle/prepare_fineweb.py \
  --out /kaggle/working/corpus \
  --dataset HuggingFaceFW/fineweb-edu \
  --config sample-10BT \
  --max-docs 4000 \
  --max-chars 40000000 \
  || echo 'FineWeb prep failed or timed out; continuing with WikiText only'


In [ ]:
#@title Optional large-model smoke test (5 minutes)
# This checks memory and TF compatibility before the 2-hour run.
!STATE_DIR=/kaggle/working/smoke \
 CORPUS_DIR=/kaggle/working/corpus \
 MODEL_SIZE=large BATCH=32 \
 EPOCHS=1 EXAMPLES=256 CORPUS_CHARS=2000000 \
 TIME_BUDGET=5 TEST_SIZE=256 \
 bash kaggle/train_kaggle.sh


In [ ]:
#@title Train for 2 hours on one GPU
!STATE_DIR=/kaggle/working/diamond_state \
 CORPUS_DIR=/kaggle/working/corpus \
 MODEL_SIZE=large \
 EPOCHS=50 \
 EXAMPLES=15000 \
 CORPUS_CHARS=50000000 \
 TIME_BUDGET=120 \
 TEST_SIZE=4096 \
 BATCH=32 \
 SENT_WEIGHT=0.8 \
 bash kaggle/train_kaggle.sh


In [ ]:
#@title Report best/final metrics
!python colab/report_metrics.py \
  /kaggle/working/diamond_state/diamond_metrics.json


## Resume and memory fallback

- Within the same Kaggle session: rerun the training cell; the engine restores `/kaggle/working/diamond_state/diamond_state.npz`.
- New session: download the state files from the notebook output and place them in `/kaggle/working/diamond_state` before training.
- Keep `CORPUS_CHARS`, `TEST_SIZE`, model dimensions, and the corpus split unchanged between sessions.
- If the large preset OOMs, rerun with `MODEL_SIZE=frontier` or `BATCH=16`.
- FineWeb-Edu is streamed; it is not downloaded as a full dataset.
